# PEM vs VWAP MR Research Notebook

Notebook ini untuk **baseline quant research** dua strategi TradingView:

- **PEM Terminator2** sebagai kandidat trend / breakout
- **VWAP Mean Reversion** sebagai kandidat range / mean reversion

Asumsi baseline:
- no look-ahead
- entry di **open bar berikutnya**
- jika SL dan TP kena pada bar yang sama: **SL first**
- satu posisi aktif per strategi
- belum ada meta-labeling
- belum ada regime switcher


In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


## 1) Config

In [ ]:
CSV_PATH = 'xauusd_m1.csv'   # ganti ke path file kamu
EXPORT_DIR = Path('outputs_pem_vwapmr')
EXPORT_DIR.mkdir(exist_ok=True, parents=True)

SPREAD_POINTS = 0.0

pem_params = {
    'atrLen': 14,
    'impulseMult': 1.1,
    'compressPct': 0.7,
    'stallBars': 2,
    'minBodyFrac': 0.20,
    'emaFastLen': 8,
    'emaSlowLen': 55,
    'use_structural_stop': True,
    'sl_buffer_points': 0.0,
    'tp_r': 2.0,
    'timeout_bars': 24,
}

vwap_params = {
    'vwapLength': 60,
    'rsiLength': 14,
    'rsiOverbought': 65,
    'rsiOversold': 25,
    'stopLossPct': 0.005,
    'enableVolFilter': True,
    'volLookback': 20,
    'volMultiplier': 3.0,
    'timeout_bars': 24,
}


## 2) Loader + indicators

In [ ]:
def load_xauusd_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    required = ['date', 'time', 'open', 'high', 'low', 'close', 'volume']
    missing = [c for c in required if c not in cols]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Found: {list(df.columns)}')

    dt = pd.to_datetime(
        df[cols['date']].astype(str) + ' ' + df[cols['time']].astype(str),
        format='%Y.%m.%d %H:%M',
        utc=True,
        errors='coerce'
    )
    out = pd.DataFrame({
        'timestamp': dt,
        'open': pd.to_numeric(df[cols['open']], errors='coerce'),
        'high': pd.to_numeric(df[cols['high']], errors='coerce'),
        'low': pd.to_numeric(df[cols['low']], errors='coerce'),
        'close': pd.to_numeric(df[cols['close']], errors='coerce'),
        'volume': pd.to_numeric(df[cols['volume']], errors='coerce').fillna(0),
    }).dropna().sort_values('timestamp').reset_index(drop=True)
    return out

def ema(s: pd.Series, length: int) -> pd.Series:
    return s.ewm(span=length, adjust=False).mean()

def sma(s: pd.Series, length: int) -> pd.Series:
    return s.rolling(length, min_periods=length).mean()

def atr(df: pd.DataFrame, length: int) -> pd.Series:
    prev_close = df['close'].shift(1)
    tr = pd.concat([
        (df['high'] - df['low']).abs(),
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(length, min_periods=length).mean()

def rsi(s: pd.Series, length: int) -> pd.Series:
    delta = s.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/length, adjust=False).mean()
    roll_down = down.ewm(alpha=1/length, adjust=False).mean()
    rs = roll_up / roll_down.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def rolling_vw_mean(src: pd.Series, vol: pd.Series, length: int) -> pd.Series:
    num = (src * vol).rolling(length, min_periods=length).sum()
    den = vol.rolling(length, min_periods=length).sum()
    return num / den.replace(0, np.nan)

def rolling_vw_abs_dev(src: pd.Series, vol: pd.Series, mean: pd.Series, length: int) -> pd.Series:
    abs_dev = (src - mean).abs() * vol
    num = abs_dev.rolling(length, min_periods=length).sum()
    den = vol.rolling(length, min_periods=length).sum()
    return num / den.replace(0, np.nan)

df = load_xauusd_csv(CSV_PATH)
df.head()


## 3) Signal builders

In [ ]:
def build_pem_signals(df: pd.DataFrame, p: Dict) -> pd.DataFrame:
    d = df.copy()
    d['atr'] = atr(d, p['atrLen'])
    d['ema_fast'] = ema(d['close'], p['emaFastLen'])
    d['ema_slow'] = ema(d['close'], p['emaSlowLen'])
    session = d['timestamp'].dt.floor('D')
    cum_pv = (d['close'] * d['volume']).groupby(session).cumsum()
    cum_v = d['volume'].groupby(session).cumsum().replace(0, np.nan)
    d['vwap'] = cum_pv / cum_v

    bar_range = d['high'] - d['low']
    body = (d['close'] - d['open']).abs()
    d['body_frac'] = np.where(bar_range > 0, body / bar_range, 0.0)
    d['strong_up'] = (d['close'] > d['open']) & (bar_range > d['atr'] * p['impulseMult'])
    d['strong_down'] = (d['close'] < d['open']) & (bar_range > d['atr'] * p['impulseMult'])
    d['small_candle'] = bar_range < d['atr'] * p['compressPct']

    state = 0
    comp_high = np.nan
    comp_low = np.nan
    stall_count = 0
    fire_long = False
    fire_short = False
    fire_bar = None

    signal = np.zeros(len(d), dtype=int)
    stop_hint = np.full(len(d), np.nan)

    for i in range(len(d)):
        if state == 0 and (bool(d.at[i, 'strong_up']) or bool(d.at[i, 'strong_down'])):
            state = 1
            stall_count = 0
            comp_high = np.nan
            comp_low = np.nan

        if state == 1 and bool(d.at[i, 'small_candle']):
            state = 2
            stall_count = 1
            comp_high = d.at[i, 'high']
            comp_low = d.at[i, 'low']
        elif state == 2 and bool(d.at[i, 'small_candle']):
            stall_count += 1
            comp_high = max(comp_high, d.at[i, 'high'])
            comp_low = min(comp_low, d.at[i, 'low'])

        valid_comp = (state == 2 and stall_count >= p['stallBars'])
        bull_break = valid_comp and (d.at[i, 'close'] > comp_high)
        bear_break = valid_comp and (d.at[i, 'close'] < comp_low)
        break_clean = d.at[i, 'body_frac'] >= p['minBodyFrac']
        fire = (bull_break or bear_break) and break_clean

        if fire:
            fire_long = bool(bull_break)
            fire_short = bool(bear_break)
            fire_bar = i
            if fire_long:
                stop_hint[i] = comp_low
            if fire_short:
                stop_hint[i] = comp_high
            state = 0
            stall_count = 0
            comp_high = np.nan
            comp_low = np.nan

        if state != 0 and stall_count > 15:
            state = 0
            stall_count = 0
            comp_high = np.nan
            comp_low = np.nan

        is_confirm_bar = (fire_bar is not None) and (i == fire_bar + 1)
        confirm_long = is_confirm_bar and fire_long and (d.at[i, 'close'] > d.at[i, 'open'])
        confirm_short = is_confirm_bar and fire_short and (d.at[i, 'close'] < d.at[i, 'open'])
        filter_long = (d.at[i, 'close'] > d.at[i, 'ema_slow']) and (d.at[i, 'close'] > d.at[i, 'vwap'])
        filter_short = (d.at[i, 'close'] < d.at[i, 'ema_slow']) and (d.at[i, 'close'] < d.at[i, 'vwap'])

        if confirm_long and filter_long:
            signal[i] = 1
        elif confirm_short and filter_short:
            signal[i] = -1

    d['signal'] = signal
    d['stop_hint'] = pd.Series(stop_hint).ffill().where(pd.Series(signal) != 0)
    return d

def build_vwapmr_signals(df: pd.DataFrame, p: Dict) -> pd.DataFrame:
    d = df.copy()
    src = d['close']
    d['basis'] = rolling_vw_mean(src, d['volume'], p['vwapLength'])
    d['dev'] = rolling_vw_abs_dev(src, d['volume'], d['basis'], p['vwapLength'])
    d['upper2'] = d['basis'] + d['dev'] * 2.0
    d['lower2'] = d['basis'] - d['dev'] * 2.0
    d['rsi'] = rsi(src, p['rsiLength'])
    d['avgVol'] = sma(d['volume'], p['volLookback'])
    d['extremeVol'] = d['volume'] > d['avgVol'] * p['volMultiplier']
    vol_condition = (~d['extremeVol']) if p['enableVolFilter'] else pd.Series(True, index=d.index)
    d['signal'] = 0
    d.loc[(src.shift(1) >= d['lower2'].shift(1)) & (src < d['lower2']) & (d['rsi'] < p['rsiOversold']) & vol_condition, 'signal'] = 1
    d.loc[(src.shift(1) <= d['upper2'].shift(1)) & (src > d['upper2']) & (d['rsi'] > p['rsiOverbought']) & vol_condition, 'signal'] = -1
    return d

pem_df = build_pem_signals(df, pem_params)
vwap_df = build_vwapmr_signals(df, vwap_params)
pem_df[['timestamp','signal']].query('signal != 0').head(), vwap_df[['timestamp','signal']].query('signal != 0').head()


## 4) Backtest engine

In [ ]:
@dataclass
class Trade:
    strategy_name: str
    direction: int
    signal_time: pd.Timestamp
    entry_time: pd.Timestamp
    exit_time: pd.Timestamp
    entry_price: float
    exit_price: float
    sl_price: float
    tp_price: float
    exit_reason: str
    r_multiple: float
    holding_bars: int

def compute_metrics(trades: pd.DataFrame) -> Dict:
    if len(trades) == 0:
        return {'total_trades': 0}
    r = trades['r_multiple'].astype(float)
    wins = r[r > 0].sum()
    losses = -r[r < 0].sum()
    equity = r.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    max_dd = dd.min()
    losing_streak = 0
    max_losing_streak = 0
    for x in r:
        if x < 0:
            losing_streak += 1
            max_losing_streak = max(max_losing_streak, losing_streak)
        else:
            losing_streak = 0
    return {
        'total_trades': int(len(trades)),
        'win_rate': float((r > 0).mean()),
        'profit_factor': float(wins / losses) if losses > 0 else float('inf'),
        'expectancy_r': float(r.mean()),
        'avg_r': float(r.mean()),
        'median_r': float(r.median()),
        'sum_r': float(r.sum()),
        'max_drawdown_r': float(max_dd),
        'max_losing_streak': int(max_losing_streak),
        'avg_holding_bars': float(trades['holding_bars'].mean()),
    }

def backtest_pem(d: pd.DataFrame, p: Dict) -> pd.DataFrame:
    trades = []
    i = 0
    while i < len(d) - 1:
        if d.at[i, 'signal'] != 0:
            direction = int(d.at[i, 'signal'])
            entry_idx = i + 1
            entry_price = d.at[entry_idx, 'open'] + (SPREAD_POINTS if direction == 1 else -SPREAD_POINTS)

            if p['use_structural_stop'] and pd.notna(d.at[i, 'stop_hint']):
                sl_price = d.at[i, 'stop_hint'] - p['sl_buffer_points'] if direction == 1 else d.at[i, 'stop_hint'] + p['sl_buffer_points']
            else:
                a = d.at[i, 'atr']
                sl_price = entry_price - a if direction == 1 else entry_price + a

            risk = abs(entry_price - sl_price)
            if pd.isna(risk) or risk <= 0:
                i += 1
                continue

            tp_price = entry_price + p['tp_r'] * risk if direction == 1 else entry_price - p['tp_r'] * risk
            exit_idx = min(entry_idx + p['timeout_bars'], len(d)-1)
            exit_price = d.at[exit_idx, 'close']
            exit_reason = 'timeout'

            for j in range(entry_idx, min(entry_idx + p['timeout_bars'] + 1, len(d))):
                hi, lo = d.at[j, 'high'], d.at[j, 'low']
                if direction == 1:
                    hit_sl, hit_tp = lo <= sl_price, hi >= tp_price
                else:
                    hit_sl, hit_tp = hi >= sl_price, lo <= tp_price

                if hit_sl and hit_tp:
                    exit_idx, exit_price, exit_reason = j, sl_price, 'sl_first_same_bar'
                    break
                elif hit_sl:
                    exit_idx, exit_price, exit_reason = j, sl_price, 'sl'
                    break
                elif hit_tp:
                    exit_idx, exit_price, exit_reason = j, tp_price, 'tp'
                    break

            r_mult = (exit_price - entry_price) / risk if direction == 1 else (entry_price - exit_price) / risk
            trades.append(Trade(
                strategy_name='PEM',
                direction=direction,
                signal_time=d.at[i, 'timestamp'],
                entry_time=d.at[entry_idx, 'timestamp'],
                exit_time=d.at[exit_idx, 'timestamp'],
                entry_price=float(entry_price),
                exit_price=float(exit_price),
                sl_price=float(sl_price),
                tp_price=float(tp_price),
                exit_reason=exit_reason,
                r_multiple=float(r_mult),
                holding_bars=int(exit_idx - entry_idx + 1),
            ))
            i = exit_idx + 1
        else:
            i += 1
    return pd.DataFrame([t.__dict__ for t in trades])

def backtest_vwapmr(d: pd.DataFrame, p: Dict) -> pd.DataFrame:
    trades = []
    i = 0
    while i < len(d) - 1:
        if d.at[i, 'signal'] != 0 and pd.notna(d.at[i, 'basis']):
            direction = int(d.at[i, 'signal'])
            entry_idx = i + 1
            entry_price = d.at[entry_idx, 'open'] + (SPREAD_POINTS if direction == 1 else -SPREAD_POINTS)
            sl_price = entry_price * (1 - p['stopLossPct']) if direction == 1 else entry_price * (1 + p['stopLossPct'])
            tp_price = d.at[i, 'basis']
            risk = abs(entry_price - sl_price)
            if pd.isna(risk) or risk <= 0:
                i += 1
                continue

            exit_idx = min(entry_idx + p['timeout_bars'], len(d)-1)
            exit_price = d.at[exit_idx, 'close']
            exit_reason = 'timeout'

            for j in range(entry_idx, min(entry_idx + p['timeout_bars'] + 1, len(d))):
                hi, lo = d.at[j, 'high'], d.at[j, 'low']
                if direction == 1:
                    hit_sl, hit_tp = lo <= sl_price, hi >= tp_price
                else:
                    hit_sl, hit_tp = hi >= sl_price, lo <= tp_price

                if hit_sl and hit_tp:
                    exit_idx, exit_price, exit_reason = j, sl_price, 'sl_first_same_bar'
                    break
                elif hit_sl:
                    exit_idx, exit_price, exit_reason = j, sl_price, 'sl'
                    break
                elif hit_tp:
                    exit_idx, exit_price, exit_reason = j, tp_price, 'tp_mean'
                    break

            r_mult = (exit_price - entry_price) / risk if direction == 1 else (entry_price - exit_price) / risk
            trades.append(Trade(
                strategy_name='VWAP_MR',
                direction=direction,
                signal_time=d.at[i, 'timestamp'],
                entry_time=d.at[entry_idx, 'timestamp'],
                exit_time=d.at[exit_idx, 'timestamp'],
                entry_price=float(entry_price),
                exit_price=float(exit_price),
                sl_price=float(sl_price),
                tp_price=float(tp_price),
                exit_reason=exit_reason,
                r_multiple=float(r_mult),
                holding_bars=int(exit_idx - entry_idx + 1),
            ))
            i = exit_idx + 1
        else:
            i += 1
    return pd.DataFrame([t.__dict__ for t in trades])


## 5) Run backtest

In [ ]:
pem_trades = backtest_pem(pem_df, pem_params)
vwap_trades = backtest_vwapmr(vwap_df, vwap_params)

pem_metrics = compute_metrics(pem_trades)
vwap_metrics = compute_metrics(vwap_trades)

print('PEM metrics')
print(json.dumps(pem_metrics, indent=2))
print()
print('VWAP MR metrics')
print(json.dumps(vwap_metrics, indent=2))


## 6) Compare + charts

In [ ]:
compare = pd.DataFrame([
    {'strategy': 'PEM', **pem_metrics},
    {'strategy': 'VWAP_MR', **vwap_metrics},
])
display(compare)

plt.figure(figsize=(12,5))
if len(pem_trades):
    plt.plot(pd.to_datetime(pem_trades['exit_time']), pem_trades['r_multiple'].cumsum(), label='PEM')
if len(vwap_trades):
    plt.plot(pd.to_datetime(vwap_trades['exit_time']), vwap_trades['r_multiple'].cumsum(), label='VWAP_MR')
plt.title('Equity Curve in R')
plt.xlabel('Exit Time')
plt.ylabel('Cumulative R')
plt.legend()
plt.grid(True)
plt.show()


## 7) Monthly breakdown + export

In [ ]:
def monthly_breakdown(trades: pd.DataFrame) -> pd.DataFrame:
    if len(trades) == 0:
        return pd.DataFrame()
    t = trades.copy()
    t['month'] = pd.to_datetime(t['exit_time']).dt.to_period('M').astype(str)
    return t.groupby('month').agg(
        trades=('r_multiple', 'size'),
        win_rate=('r_multiple', lambda x: (x > 0).mean()),
        sum_r=('r_multiple', 'sum'),
        avg_r=('r_multiple', 'mean'),
        median_r=('r_multiple', 'median'),
    ).reset_index()

pem_monthly = monthly_breakdown(pem_trades)
vwap_monthly = monthly_breakdown(vwap_trades)

display(pem_monthly.tail(12))
display(vwap_monthly.tail(12))

pem_trades.to_csv(EXPORT_DIR / 'pem_trades.csv', index=False)
vwap_trades.to_csv(EXPORT_DIR / 'vwapmr_trades.csv', index=False)
pem_monthly.to_csv(EXPORT_DIR / 'pem_monthly.csv', index=False)
vwap_monthly.to_csv(EXPORT_DIR / 'vwapmr_monthly.csv', index=False)
with open(EXPORT_DIR / 'metrics_compare.json', 'w') as f:
    json.dump({'PEM': pem_metrics, 'VWAP_MR': vwap_metrics}, f, indent=2)

print('Saved to:', EXPORT_DIR.resolve())
